In [7]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [ ]:
with open('C:\\Users\\dylan\\OneDrive\\Documentos\\mensajeria\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

In [ ]:
dim_proveedor=pd.read_sql_table('dim_proveedor', etl_conn)
dim_mensajero=pd.read_sql_table('dim_mensajero', etl_conn)
dim_sede=pd.read_sql_table('dim_sede', etl_conn)
dim_hora=pd.read_sql_table('dim_hora', etl_conn)    
dim_geografia=pd.read_sql_table('dim_geografia', etl_conn)
dim_fecha=pd.read_sql_table('dim_fecha', etl_conn)
dim_usuario=pd.read_sql_table('dim_usuario', etl_conn)
trans_servicio=pd.read_sql_table('trans_servicio', etl_conn)

tabla_sede=pd.read_sql_table('clientes_usuarioaquitoy', mensajeria)
tabla_estados=pd.read_sql_table('mensajeria_estadosservicio', mensajeria)

In [ ]:
hecho_servicios= trans_servicio.merge(dim_proveedor[['id_proveedor','key_proveedor']], left_on='cliente_id',right_on='id_proveedor', how='left') \
    .merge(dim_mensajero[['id_operaciones','key_mensajero']], left_on='mensajero_id',right_on='id_operaciones', how='left') \
    .merge(dim_mensajero[['id_operaciones','key_mensajero']], left_on='mensajero2_id', right_on='id_operaciones', how='left') \
    .merge(dim_mensajero[['id_operaciones','key_mensajero']], left_on='mensajero3_id', right_on='id_operaciones', how='left') \
    .merge(dim_geografia[['ciudad_id','key_geografia']], left_on='ciudad_origen_id',right_on='ciudad_id', how='left') \
    .merge(dim_geografia[['ciudad_id','key_geografia']], left_on='ciudad_destino_id',right_on='ciudad_id', how='left') \
    

hecho_servicios.drop(columns=['id_proveedor','id_operaciones_x','id_operaciones_y','id_operaciones','ciudad_id_x','ciudad_id_y',
                              'mensajero_id','mensajero2_id','mensajero3_id','ciudad_origen_id','ciudad_destino_id',
                              ], inplace=True)
hecho_servicios.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 346 entries, 0 to 345
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id                  346 non-null    int64         
 1   fecha_solicitud     346 non-null    datetime64[ns]
 2   hora_solicitud      346 non-null    object        
 3   cliente_id          346 non-null    int64         
 4   destino_id          346 non-null    int64         
 5   origen_id           346 non-null    int64         
 6   tipo_servicio_id    346 non-null    int64         
 7   tipo_vehiculo_id    346 non-null    int64         
 8   usuario_id          346 non-null    int64         
 9   ciudad_destino_id   346 non-null    int64         
 10  ciudad_origen_id    346 non-null    int64         
 11  key_trans_servicio  346 non-null    int64         
 12  key_proveedor       346 non-null    int64         
 13  key_mensajero_x     10 non-null     float64       

In [21]:
hecho_servicios.drop(columns=['id_proveedor','id_operaciones_x','id_operaciones_y','id_operaciones','ciudad_id_x','ciudad_id_y'], inplace=True)